# Comparison SSL Phase 2: BIOT Fine-Tuning (Hosseini)

## Overview
This notebook performs supervised fine-tuning on the **Hosseini (Indian)** dataset using the pre-trained **BIOT** encoder. 
We load the weights generated in Phase 1 and adapt the model to classify Stress vs. Baseline states.

In [ ]:
# ── CELL 1: Setup and Architecture ──────────────────────────────────────────
!pip install -q xlrd
import os, glob, pickle, torch, torch.nn as nn, numpy as np, pandas as pd
from scipy.signal import resample
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from einops import rearrange
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
indian_dirs = glob.glob('/kaggle/input/**/datasetindiani', recursive=True)
INDIAN_RAW_DIR = indian_dirs[0] if indian_dirs else '/kaggle/input/datasetindiani'
WEIGHTS_PATH = glob.glob('/kaggle/input/**/biot_encoder_pretrained.pt', recursive=True)
WEIGHTS_PATH = WEIGHTS_PATH[0] if WEIGHTS_PATH else '/kaggle/working/biot_encoder_pretrained.pt'

class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class BIOTEncoder(nn.Module):
    def __init__(self, n_channels=6, seg_len=640, chunk_size=64, d_model=256, n_heads=8, n_layers=4, dropout=0.1):
        super().__init__()
        self.n_chunks, self.chunk_size, self.d_model = seg_len // chunk_size, chunk_size, d_model
        self.patch_embed = nn.Linear(chunk_size, d_model); self.channel_emb = nn.Embedding(n_channels, d_model)
        self.pos_enc = SinusoidalPE(d_model, max_len=self.n_chunks * n_channels + 1); self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers); self.norm = nn.LayerNorm(d_model)
    def forward(self, x):
        B, C, T = x.shape
        x = rearrange(x[:, :, :self.n_chunks * self.chunk_size], 'b c (n p) -> b c n p', p=self.chunk_size)
        x = self.patch_embed(x) + self.channel_emb(torch.arange(C, device=x.device)).unsqueeze(0).unsqueeze(2)
        x = torch.cat([self.cls_token.expand(B, -1, -1), rearrange(x, 'b c n d -> b (c n) d')], dim=1)
        return self.norm(self.transformer(self.pos_enc(x)))[:, 0, :]

class BIOTClassifier(nn.Module):
    def __init__(self, encoder, n_classes=2, dropout=0.5):
        super().__init__()
        self.encoder = encoder; self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(encoder.d_model, encoder.d_model // 2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(encoder.d_model // 2, n_classes))
    def forward(self, x): return self.head(self.encoder(x.float()))

### 2. Loading Pre-trained Weights

In [ ]:
encoder = BIOTEncoder()
if os.path.exists(WEIGHTS_PATH):
    encoder.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
    print(f"Successfully loaded pre-trained weights from: {WEIGHTS_PATH}")
else:
    print("WARNING: Pre-trained weights not found. Training from scratch!")
model = BIOTClassifier(encoder).to(device)

### 3. Binary-Safe Data Reading

In [ ]:
def final_read(fp, cols):
    with open(fp, 'rb') as f: magic = f.read(8)
    if magic.startswith(b'\xd0\xcf\x11\xe0\xa1\xb1'):
        try: return pd.read_excel(fp, header=None, skiprows=2, usecols=cols, engine='xlrd').astype(np.float32)
        except: pass
    for enc in ['latin1', 'utf-8-sig']:
        try: return pd.read_csv(fp, header=None, skiprows=2, usecols=cols, dtype=np.float32, on_bad_lines='skip', encoding=enc)
        except: continue
    return pd.DataFrame()

TARGET_LEN, CHANNELS = 640, ['ACC_x', 'ACC_y', 'ACC_z', 'BVP', 'EDA', 'TEMP']
FREQ = {'BVP': 64, 'EDA': 4, 'TEMP': 4, 'ACC_x': 32, 'ACC_y': 32, 'ACC_z': 32}
TASK_MAP = {1: 1, 2: 1, 3: 1, 4: 0, 5: 0}

all_X, all_y, all_subs = [], [], []
subs = sorted([d for d in os.listdir(INDIAN_RAW_DIR) if d.lower().startswith('subject_')])

for sub_folder in tqdm(subs):
    tasks_found = {}
    for root, dirs, files in os.walk(os.path.join(INDIAN_RAW_DIR, sub_folder)):
        for f in files:
            if not f.lower().endswith('.csv'): continue
            f_up = f.upper(); f_p = os.path.join(root, f)
            b = 'ACC' if 'ACC' in f_up else 'BVP' if any(k in f_up for k in ['BVP','PLETH','PULSE']) else 'EDA' if any(k in f_up for k in ['EDA','GSR']) else 'TEMP' if 'TEMP' in f_up else None
            if b:
                tid = next((t for t in TASK_MAP if f'T{t}' in f_up or f'T{t}' in root.upper() or f'_{t}' in f_up), 1)
                if tid not in tasks_found: tasks_found[tid] = {s: [] for s in ['ACC', 'BVP', 'EDA', 'TEMP']}
                tasks_found[tid][b].append(f_p)
    for tid, sensors in tasks_found.items():
        if not sensors['ACC']: continue
        data = {}
        for base, f_list in sensors.items():
            if not f_list: continue
            df = pd.concat([final_read(fp, ([0,1,2] if base=='ACC' else [0])) for fp in sorted(f_list)], ignore_index=True)
            if df.empty: continue
            if base == 'ACC': data['ACC_x'], data['ACC_y'], data['ACC_z'] = df.iloc[:,0].values, df.iloc[:,1].values, df.iloc[:,2].values
            else: data[base] = df.iloc[:,0].values
        if 'ACC_x' not in data: continue
        for c in CHANNELS: 
            if c not in data: data[c] = np.zeros(int(len(data['ACC_x']) * FREQ[c] / FREQ['ACC_x']), dtype=np.float32)
        dur = min([len(data[c])/FREQ[c] for c in CHANNELS if len(data[c])>0])
        for start in range(0, int(dur)-10+1, 10):
            segs = [resample(data[c][int(start*FREQ[c]):int((start+10)*FREQ[c])], TARGET_LEN) for c in CHANNELS]
            segs = [(s - s.mean())/(s.std()+1e-8) for s in segs]
            all_X.append(np.stack(segs)); all_y.append(TASK_MAP[tid]); all_subs.append(sub_folder)

### 4. Training and Evaluation

In [ ]:
u_subs = np.unique(all_subs); np.random.seed(42); np.random.shuffle(u_subs)
tr_s, te_s = u_subs[:int(0.7*len(u_subs))], u_subs[int(0.7*len(u_subs)):]
tr_idx = np.isin(all_subs, tr_s)

class SigDataset(Dataset):
    def __init__(self, m): self.X, self.y = torch.tensor(np.array(all_X)[m]), torch.tensor(np.array(all_y)[m])
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

weights = 1. / np.array([len(np.where(np.array(all_y)[tr_idx] == t)[0]) for t in np.unique(np.array(all_y)[tr_idx])])
sampler = WeightedRandomSampler(torch.tensor([weights[int(t)] for t in np.array(all_y)[tr_idx]]), len(np.where(tr_idx)[0]))
train_loader = DataLoader(SigDataset(tr_idx), batch_size=32, sampler=sampler)
test_loader = DataLoader(SigDataset(np.isin(all_subs, te_s)), batch_size=32)

opt = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.1); crit = nn.CrossEntropyLoss()
for e in range(1, 51):
    model.train()
    for xb, yb in train_loader: opt.zero_grad(); crit(model(xb.float().to(device)), yb.to(device)).backward(); opt.step()

model.eval(); p, t = [], []
with torch.no_grad():
    for xb, yb in test_loader: p.extend(model(xb.float().to(device)).argmax(1).cpu().numpy()); t.extend(yb.numpy())
print("Final Benchmarking:"); print(classification_report(t, p, target_names=['Baseline', 'Stress']))
acc = accuracy_score(t, p)
f1_stress = f1_score(t, p)
f1_macro = f1_score(t, p, average='macro')

import pandas as pd
df = pd.DataFrame({
    'Modello': ['Corponi (Master)', 'BIOT (Pre-trained)'],
    'Accuracy': ['57.65%', f'{acc*100:.2f}%'],
    'F1-Score (Stress)': ['70.37%', f'{f1_stress*100:.2f}%'],
    'Macro F1': ['60.00%', f'{f1_macro*100:.2f}%']
})
print('\n--- CONFRONTO FINALE HOSSEINI ---')
print(df.to_string(index=False))

# --- Aggiunta Matrice di Confusione ---
if 'confusion_matrix' not in locals() and 'confusion_matrix' not in globals():
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    import matplotlib.pyplot as plt

cm = confusion_matrix(t, p)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Baseline', 'Stress'])
disp.plot(cmap='Blues')
plt.title('Matrice di Confusione')
plt.show()
